# RAG 베이스라인 재현

- 목적: 전처리 완료된 `processed_documents.jsonl`을 입력으로 받아 baseline RAG를 재현합니다.
- 포함 범위: 청킹, 임베딩/Chroma 적재, 샘플 질의, Day3 baseline 평가, 자동 judge
- 제외 범위: RFP 파싱 원본 생성 단계


In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()  # 필요 시 리포지토리 루트 절대경로로 수정
PROCESSED_DOCS_PATH = PROJECT_ROOT / "processed_data" / "processed_documents.jsonl"
QUESTION_SET_PATH = PROJECT_ROOT / "evaluation" / "day3_partA_eval_questions_v1.txt"
CHROMA_DIR = PROJECT_ROOT / "rag_outputs" / "chroma_db"
RAG_OUTPUTS = PROJECT_ROOT / "rag_outputs"

PYTHON = sys.executable

def run_cmd(args, cwd=PROJECT_ROOT):
    print("[RUN]", " ".join(str(x) for x in args))
    result = subprocess.run(
        [str(x) for x in args],
        cwd=str(cwd),
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"command failed: {result.returncode}")
    return result

def show_csv(path, n=20):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    display(df.head(n))
    return df

print("PROJECT_ROOT =", PROJECT_ROOT)
print("CHROMA_DIR   =", CHROMA_DIR)



In [ ]:
required_paths = [PROCESSED_DOCS_PATH, QUESTION_SET_PATH, PROJECT_ROOT / '.env']
for path in required_paths:
    print(path, '->', path.exists())


## 1. Contextual Chunking

In [ ]:
BASELINE_CHUNK_PATH = RAG_OUTPUTS / 'contextual_chunks.jsonl'
BASELINE_CHUNK_SUMMARY = RAG_OUTPUTS / 'contextual_chunks_summary.csv'
run_cmd([
    PYTHON, PROJECT_ROOT / '01_컨텍스트_청킹.py',
    '--입력경로', PROCESSED_DOCS_PATH,
    '--출력경로', BASELINE_CHUNK_PATH,
    '--요약CSV경로', BASELINE_CHUNK_SUMMARY,
    '--청크크기', '500',
    '--겹침크기', '80',
])
show_csv(BASELINE_CHUNK_SUMMARY)


## 2. Embedding + Chroma 적재

In [ ]:
BASELINE_COLLECTION = 'rfp_contextual_chunks_v1'
run_cmd([
    PYTHON, PROJECT_ROOT / '02_임베딩_생성_크로마적재.py',
    '--청크경로', BASELINE_CHUNK_PATH,
    '--크로마경로', CHROMA_DIR,
    '--컬렉션이름', BASELINE_COLLECTION,
    '--기존컬렉션초기화',
])
show_csv(RAG_OUTPUTS / 'chroma_적재_요약.csv')


## 3. 샘플 질의

In [ ]:
SAMPLE_QUERY = '이 프로젝트의 주요 요구사항과 구축 범위를 요약해줘'
run_cmd([
    PYTHON, PROJECT_ROOT / '03_나이브_RAG_베이스라인.py',
    '--질문', SAMPLE_QUERY,
    '--상위개수', '5',
    '--크로마경로', CHROMA_DIR,
    '--컬렉션이름', BASELINE_COLLECTION,
    '--응답모델', 'gpt-5-mini',
])


## 4. Day3 Baseline 평가

In [ ]:
BASELINE_EVAL_DIR = RAG_OUTPUTS / 'baseline_eval_notebook'
run_cmd([
    PYTHON, PROJECT_ROOT / '04_베이스라인_평가.py',
    '--질문셋경로', QUESTION_SET_PATH,
    '--크로마경로', CHROMA_DIR,
    '--컬렉션이름', BASELINE_COLLECTION,
    '--응답모델', 'gpt-5-mini',
], cwd=PROJECT_ROOT)
show_csv(PROJECT_ROOT / 'rag_outputs' / 'baseline_eval' / 'baseline_eval_summary.csv')


## 5. 자동 Judge

In [ ]:
run_cmd([
    PYTHON, PROJECT_ROOT / '05_자동_Judge_채점.py',
    '--평가디렉토리', PROJECT_ROOT / 'rag_outputs' / 'baseline_eval',
    '--judge모델', 'gpt-5',
    '--크로마경로', CHROMA_DIR,
    '--컬렉션이름', BASELINE_COLLECTION,
    '--resume',
])
show_csv(PROJECT_ROOT / 'rag_outputs' / 'baseline_eval' / 'baseline_eval_manual_summary.csv')
